# 05 · Evaluación — Dry Bean Dataset
**CRISP-DM fase 5: Evaluación** · Sprint 3 · Backlog: PB-05, PB-06


| **Entrada** | `metricas_modelos.csv`, `predicciones_test.csv` — resultados de la etapa de modelado |
| **Entrada** | `modelo_logistica.joblib`, `modelo_xgboost.joblib` — modelos candidatos |
| **Salida** | `modelo_final.joblib` / `modelo_final.pkl` — modelo aprobado para despliegue |

En el modelado, la regresión logística y XGBoost quedaron **técnicamente empatados**. Los dos pasan a esta fase, que responde una pregunta distinta: **¿cuál le sirve a la empresa y por qué?**

Se siguen las siguientes tareas: **evaluar los resultados frente al negocio, revisar el proceso y decidir los siguientes pasos**. No se entrenan modelos ni se recalculan resultados del modelado.

## 0. Entorno y datos

In [2]:
import time
import shutil
import pickle
from pathlib import Path

import pandas as pd
import joblib
from sklearn.metrics import recall_score

from google.colab import drive
drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/"

metricas = pd.read_csv(RUTA + "metricas_modelos.csv", index_col="modelo")
pred = pd.read_csv(RUTA + "predicciones_test.csv")
modelos = {
    "Regresión logística": joblib.load(RUTA + "modelo_logistica.joblib"),
    "XGBoost": joblib.load(RUTA + "modelo_xgboost.joblib"),
}
COL_PRED = {"Regresión logística": "pred_logistica", "XGBoost": "pred_xgboost"}
CLASES = sorted(pred["real"].unique())

print("Granos de prueba:", len(pred))
metricas.round(4)

Mounted at /content/drive
Granos de prueba: 2709


,accuracy,balanced_accuracy,f1_macro,precision_macro,recall_macro,cohen_kappa,log_loss,f1_macro_media,f1_macro_desv,accuracy_media,balanced_acc_media
modelo,,,,,,,,,,,
Dummy (clase más frecuente),0.2617,0.1429,0.0593,0.0374,0.1429,0.0000,NaN,0.0593,0.0000,0.2619,0.1429
Regresión logística,0.9151,0.9292,0.9277,0.9270,0.9292,0.8975,0.2236,0.9330,0.0055,0.9194,0.9339
XGBoost,0.9169,0.9289,0.9292,0.9296,0.9289,0.8996,0.2599,0.9381,0.0042,0.9260,0.9378


---
# Tarea 1 · Evaluar los resultados frente al negocio

## 1.1 Criterio de éxito
En la comprensión del negocio se definió como criterio de éxito un **F1 macro alto**, porque todas las variedades importan por igual. Para poder decir "cumple / no cumple", el Product Owner del equipo (John Carmona), en representación de la empresa, lo concretó en tres umbrales para aceptar un modelo que reemplace la clasificación manual:

| Criterio | Umbral | Por qué |
| --- | --- | --- |
| F1 macro en prueba | ≥ 0,90 | Nivel mínimo para que la clasificación automática sea confiable en todas las variedades |
| Recall mínimo por clase | ≥ 0,85 | Ninguna variedad puede quedar mal clasificada en más del 15 % de sus granos |
| Desviación del F1 macro en validación cruzada | ≤ 0,01 | El desempeño no debe depender de qué granos se usen para entrenar |



In [3]:
UMBRALES = {"f1_macro": 0.90, "recall_minimo": 0.85, "desviacion_cv": 0.01}

filas = []
for nombre, col in COL_PRED.items():
    recalls = recall_score(pred["real"], pred[col], average=None, labels=CLASES)
    filas.append({
        "modelo": nombre,
        "f1_macro": metricas.loc[nombre, "f1_macro"],
        "recall_minimo": recalls.min(),
        "clase_recall_minimo": CLASES[recalls.argmin()],
        "desviacion_cv": metricas.loc[nombre, "f1_macro_desv"],
    })
criterios = pd.DataFrame(filas).set_index("modelo")

criterios["cumple_f1"] = criterios["f1_macro"] >= UMBRALES["f1_macro"]
criterios["cumple_recall"] = criterios["recall_minimo"] >= UMBRALES["recall_minimo"]
criterios["cumple_estabilidad"] = criterios["desviacion_cv"] <= UMBRALES["desviacion_cv"]
criterios["cumple_todo"] = criterios[["cumple_f1", "cumple_recall", "cumple_estabilidad"]].all(axis=1)
criterios.round(4)

,f1_macro,recall_minimo,clase_recall_minimo,desviacion_cv,cumple_f1,cumple_recall,cumple_estabilidad,cumple_todo
modelo,,,,,,,,
Regresión logística,0.9277,0.8787,DERMASON,0.0055,True,True,True,True
XGBoost,0.9292,0.8691,SIRA,0.0042,True,True,True,True


**Lectura:** los dos modelos cumplen los tres criterios de éxito:
- **F1 macro:** 0,928 la regresión logística y 0,929 XGBoost, ambos por encima del mínimo de 0,90.
- **Recall mínimo por clase:** en la logística, la variedad más difícil es DERMASON (0,879); en XGBoost, SIRA (0,869). Las dos superan el umbral de 0,85, y la de XGBoost queda más cerca del límite.
- **Estabilidad:** la desviación en validación cruzada es 0,0055 para la logística y 0,0042 para XGBoost, ambas muy por debajo del máximo de 0,01.

**El criterio de éxito no desempata:** los dos modelos son aceptables para el negocio, así que la decisión depende de otros factores, que se analizan en las secciones siguientes.

## 1.2 Los errores en lenguaje de negocio
Para cada variedad: de cada **1.000 granos**, cuántos se clasifican bien y con qué otra variedad se confunden más.

In [4]:
def errores_por_mil(col):
    filas = []
    for clase in CLASES:
        granos = pred[pred["real"] == clase]
        errados = granos[granos[col] != clase][col].value_counts()
        filas.append({
            "variedad": clase,
            "bien_de_cada_1000": round((granos[col] == clase).mean() * 1000),
            "confundida_mas_con": errados.index[0] if len(errados) else "—",
            "de_cada_1000": round(errados.iloc[0] / len(granos) * 1000) if len(errados) else 0,
        })
    return pd.DataFrame(filas).set_index("variedad")

pd.concat({nombre: errores_por_mil(col) for nombre, col in COL_PRED.items()}, axis=1)

Regresión logística                                  \
           bien_de_cada_1000 confundida_mas_con de_cada_1000   
variedad                                                       
BARBUNYA                 898               CALI           60   
BOMBAY                  1000                  —            0   
CALI                     945           BARBUNYA           28   
DERMASON                 879               SIRA          103   
HOROZ                    944               CALI           32   
SEKER                    946           BARBUNYA           22   
SIRA                     894           DERMASON           57   

                   XGBoost                                  
         bien_de_cada_1000 confundida_mas_con de_cada_1000  
variedad                                                    
BARBUNYA               902               CALI           64  
BOMBAY                1000                  —            0  
CALI                   936           BARBUNYA           34  
DERMASON               907               SIRA           73  
HOROZ                  941               SIRA           22  
SEKER                  948               SIRA           25  
SIRA                   869           DERMASON           95

**Lectura:**
- **BOMBAY** se clasifica perfecto con los dos modelos (1.000 de cada 1.000).
- **HOROZ, SEKER y CALI** quedan bien con ambos: entre 936 y 948 granos de cada 1.000 bien clasificados.
- **BARBUNYA** se confunde sobre todo con CALI (60 a 64 de cada 1.000), las dos variedades grandes y de forma parecida.
- El punto débil de los dos modelos es el par **DERMASON / SIRA**, pero con un sesgo opuesto:
  - La **regresión logística** se equivoca más con DERMASON: 103 de cada 1.000 granos DERMASON salen etiquetados como SIRA. En cambio, acierta más con SIRA (894 de cada 1.000).
  - **XGBoost** se equivoca más con SIRA: 95 de cada 1.000 granos SIRA salen etiquetados como DERMASON. En cambio, acierta más con DERMASON (907 de cada 1.000).
- **Pregunta para el negocio:** ¿qué es más costoso, vender DERMASON como SIRA o SIRA como DERMASON? Si alguna de las dos variedades tiene mayor precio o exigencias de calidad distintas, ese dato podría inclinar la decisión hacia uno u otro modelo. Por ahora no se conoce, así que los dos errores se consideran igual de costosos.

## 1.3 ¿Los dos modelos fallan en los mismos granos?
Si la mayoría de los errores coinciden, XGBoost no "ve" nada que la logística no vea: su complejidad extra no aporta información nueva.

In [5]:
mal_log = pred["pred_logistica"] != pred["real"]
mal_xgb = pred["pred_xgboost"] != pred["real"]

coincidencia = pd.Series({
    "granos de prueba": len(pred),
    "fallan los dos": (mal_log & mal_xgb).sum(),
    "falla solo la logística": (mal_log & ~mal_xgb).sum(),
    "falla solo XGBoost": (~mal_log & mal_xgb).sum(),
    "aciertan los dos": (~mal_log & ~mal_xgb).sum(),
})
print(coincidencia.to_string())
print(f"\nLos dos modelos dan la misma predicción en el {(pred['pred_logistica'] == pred['pred_xgboost']).mean():.1%} de los granos.")
print(f"De los errores de la logística, el {(mal_log & mal_xgb).sum() / mal_log.sum():.0%} también los comete XGBoost.")

granos de prueba           2709
fallan los dos              156
falla solo la logística      74
falla solo XGBoost           69
aciertan los dos           2410

Los dos modelos dan la misma predicción en el 94.3% de los granos.
De los errores de la logística, el 68% también los comete XGBoost.


**Lectura:** los dos modelos dan la misma predicción en el 94,3 % de los granos de prueba. De los 230 errores de la regresión logística, 156 (68 %) son granos en los que XGBoost también se equivoca; del lado de XGBoost, esos 156 son el 69 % de sus 225 errores. Los errores que no comparten casi se compensan: 74 granos que solo falla la logística frente a 69 que solo falla XGBoost.

En otras palabras, XGBoost no "ve" nada que la logística no vea: los dos tropiezan con los mismos granos difíciles. Esto refuerza que el techo de desempeño lo ponen los datos (variedades de tamaño y forma muy parecidos), no el algoritmo.

## 1.4 Costo de operación
Al negocio también le importa cuánto cuesta mantener y usar el modelo.

In [6]:
X_prueba = pred.drop(columns=["real", "pred_logistica", "pred_xgboost"])
archivos = {"Regresión logística": "modelo_logistica.joblib", "XGBoost": "modelo_xgboost.joblib"}

filas = []
for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    for _ in range(20):
        modelo.predict(X_prueba)
    ms_por_1000 = (time.perf_counter() - inicio) / 20 / len(X_prueba) * 1000 * 1000
    filas.append({
        "modelo": nombre,
        "tamano_archivo_KB": Path(RUTA + archivos[nombre]).stat().st_size / 1024,
        "ms_por_1000_granos": ms_por_1000,
    })
pd.DataFrame(filas).set_index("modelo").round(1)

,tamano_archivo_KB,ms_por_1000_granos
modelo,,
Regresión logística,267.9,1.5
XGBoost,1690.2,32.8


**Lectura**: la logística ocupa varias veces menos espacio y predice varias veces más rápido. Las dos son rápidas para una línea de clasificación (milisegundos por cada 1.000 granos), así que la velocidad no es decisiva, pero suma a favor del modelo simple.

**Otros factores que no se miden con código:**

| Factor | Regresión logística | XGBoost |
| --- | --- | --- |
| Explicabilidad | Coeficientes por variable, fáciles de explicar | Cientos de árboles; requiere herramientas extra para explicarlo |
| Dependencias | Solo scikit-learn | Requiere la librería `xgboost` |
| Riesgo de sobreajuste | Bajo (brecha ≈ 0,007) | Alto (brecha ≈ 0,07) |
| Mantenimiento | Simple de reentrenar y auditar | Más hiperparámetros que vigilar |

## 1.5 Resumen de la evaluación frente al negocio

| Aspecto | Regresión logística | XGBoost | Ventaja |
| --- | --- | --- | --- |
| Criterio de éxito | Cumple | Cumple | Empate |
| F1 macro en prueba | ≈ 0,928 | ≈ 0,929 | Empate |
| Errores principales | DERMASON → SIRA | SIRA → DERMASON | Depende del costo (dato desconocido) |
| Errores compartidos | ≈ 2/3 de sus errores son comunes | | Empate |
| Sobreajuste | Bajo | Alto | Logística |
| Costo y explicabilidad | Menor | Mayor | Logística |

**Conclusión:** con desempeño equivalente, la **regresión logística** ofrece lo mismo con menos riesgo, menos costo y más facilidad para explicarla al negocio. Se confirma la recomendación técnica del modelado.

---
# Tarea 2 · Revisar el proceso

**Lo que se hizo bien:**
- Las fases quedaron separadas y trazables: un notebook por fase de CRISP-DM, conectados por archivos.
- No hubo fuga de información: la partición se hizo antes de SMOTE y del escalado, y ambos se aplicaron dentro de un pipeline, también en la validación cruzada.
- Los duplicados se detectaron en el EDA y se eliminaron en la preparación.
- La receta reutilizada no se aceptó tal cual: se reentrenó con nuestra partición y se descubrió un sobreajuste que el autor no reporta.
- Se comparó contra un baseline y contra un piso (Dummy).

**Limitaciones y riesgos:**
- **Solo variables geométricas:** el dataset no incluye color ni textura, que en la práctica distinguen varias variedades. Esto explica el techo de ≈ 92-93 % y los errores DERMASON / SIRA.
- **Medidas en píxeles:** dependen de la cámara, la distancia y la resolución. Si la planta usa otro montaje, el modelo podría fallar sin aviso.
- **Una sola fuente de datos**, de un solo estudio. No se ha probado con granos de otras cosechas ni de otros lugares.
- **Hiperparámetros sin ajustar:** se usaron los del autor (valores por defecto de XGBoost) y los de la guía para la logística.
- **Criterio de éxito supuesto:** los umbrales y el costo de cada error los fijó el equipo, no el negocio.

---
# Tarea 3 · Decidir los siguientes pasos

## 3.1 Acciones posibles

| Acción | Cuándo tendría sentido |
| --- | --- |
| **Desplegar la logística como piloto** | Ya: cumple el criterio de éxito |
| Confirmar con la empresa el costo de cada tipo de error | Antes de producción: podría cambiar la decisión (sección 1.2) |
| Validar con imágenes de la cámara de la planta | Antes de producción: riesgo de las medidas en píxeles |
| Agregar variables de color o textura | Siguiente iteración: es lo que más podría mejorar DERMASON / SIRA |
| Ajustar XGBoost (menos árboles, early stopping) | Solo si se agregan variables nuevas y vuelve a valer la pena un modelo complejo |
| Monitorear el desempeño en producción y reentrenar | Durante la operación |

## 3.2 Decisión
**Se aprueba la regresión logística** para pasar a despliegue como piloto. Cumple el criterio de éxito, rinde igual que XGBoost, no sobreajusta y es más simple de operar y explicar. XGBoost se conserva como registro del experimento.

In [7]:
MODELO_APROBADO = "Regresión logística"

shutil.copy(RUTA + archivos[MODELO_APROBADO], RUTA + "modelo_final.joblib")
with open(RUTA + "modelo_final.pkl", "wb") as f:
    pickle.dump(modelos[MODELO_APROBADO], f)

print("Modelo aprobado:", MODELO_APROBADO)
print("Guardado como modelo_final.joblib y modelo_final.pkl en", RUTA)

Modelo aprobado: Regresión logística
Guardado como modelo_final.joblib y modelo_final.pkl en /content/drive/MyDrive/


El archivo `modelo_final.pkl` es el que se usará en la fase de **Despliegue**. Así nadie despliega por error el modelo que no se aprobó.